# Pipelines de stream

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Montar um pipeline de stream: uma origem, etapas como filter, map, sorted e limit, e depois uma operação terminal.
- Calcular com streams de números (sum, average, max) e tratar um resultado vazio com Optional.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Um pipeline de stream
2. map, sorted e toList
3. Streams de números e intervalos
4. Operações terminais: obtendo uma resposta
5. Streams são preguiçosos e de uso único
6. Laços ou streams?

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Um pipeline de stream

Um **stream** processa uma coleção como uma linha de montagem: uma **origem** (`lista.stream()`), quantas **etapas intermediárias** quiser, cada uma devolvendo um stream novo, e **uma operação terminal** que produz o resultado.

- `filter(teste)` mantém os itens que passam no teste;
- `map(f)` transforma cada item em outro;
- `sorted()` os põe em ordem;
- `sum()`, `count()`, `toList()`… encerram o pipeline.

Os slides acompanham `[3, 8, 5, 12]` passando por `filter`, `map` e `sum`.

#### 🎞️ Um pipeline de stream, etapa por etapa

![Stream.of(3, 8, 5, 12): o filter mantém os valores acima de 4, o map os dobra para 16, 10, 24, e o sum dá 50](stream-steps.pt-br.svg)

In [ ]:
List<Integer> values = List.of(3, 8, 5, 12);
int result = values.stream()
        .filter(x -> x > 4)
        .mapToInt(x -> x * 2)
        .sum();
IO.println(result);
IO.println(values);   // a lista de origem não mudou

🧪 **Experimente:** tire a linha do `filter` e preveja a soma antes. Depois mantenha só os pares em vez dos maiores que 4.

### ✏️ Exercício: Conte as palavras longas

Conte as palavras com pelo menos `min` letras, com um stream: filter e depois count.

In [ ]:
long countLong(List<String> words, int min) {
    // TODO: words.stream().filter(...).count()
    return 0;
}

void main() {
    check("duas longas", countLong(List.of("a", "stream", "of", "words"), 4), 2L);
    check("nenhuma", countLong(List.of("a", "b"), 3), 0L);
}

<details>
<summary>Ver uma resposta</summary>

```java
long countLong(List<String> words, int min) {
    return words.stream().filter(w -> w.length() >= min).count();
}

void main() {
    check("duas longas", countLong(List.of("a", "stream", "of", "words"), 4), 2L);
    check("nenhuma", countLong(List.of("a", "b"), 3), 0L);
}
```

</details>

## 2. map, sorted e toList

O `map` transforma cada item em outro, até de outro tipo (uma palavra no tamanho dela). O `sorted()` usa a ordem natural, ou um comparador que você passar. O `toList()` junta o resultado numa lista nova. O `distinct()` tira os repetidos, e o `limit(n)` fica com os n primeiros.

In [ ]:
List<Integer> readings = List.of(12, -3, 7, 3, 0, -12);
List<Integer> sizes = readings.stream()
        .map(Math::abs)          // the size of each reading, without its sign
        .distinct()
        .sorted()
        .toList();
IO.println(sizes);
List<String> labels = sizes.stream().map(n -> "#" + n).toList();
IO.println(labels);

🧪 **Experimente:** mova o `distinct()` para antes da linha do `map`: por que 3 e 12 aparecem duas vezes agora? Depois acrescente `.limit(2)`.

### ✏️ Exercício: Grite cada palavra

Devolva uma lista com cada palavra em maiúsculas, com um stream: map e depois toList.

In [ ]:
List<String> shout(List<String> words) {
    // TODO: mapeie cada palavra com w -> w.toUpperCase(Locale.ROOT)
    return List.of();
}

void main() {
    check("duas palavras", shout(List.of("hi", "there")), List.of("HI", "THERE"));
    check("vazio", shout(List.of()), List.of());
}

<details>
<summary>Ver uma resposta</summary>

```java
List<String> shout(List<String> words) {
    return words.stream().map(w -> w.toUpperCase(Locale.ROOT)).toList();   // ROOT: the same in every country
}

void main() {
    check("duas palavras", shout(List.of("hi", "there")), List.of("HI", "THERE"));
    check("vazio", shout(List.of()), List.of());
}
```

</details>

## 3. Streams de números e intervalos

`mapToInt` e `mapToDouble` transformam um stream de objetos num stream de números, que ganha `sum`, `average`, `min` e `max`. `IntStream.range(a, b)` dá os números de `a` até `b`, sem incluir `b` (`rangeClosed` inclui `b`): um contador de laço como stream.

In [ ]:
List<String> words = List.of("stream", "map", "filter");
int letters = words.stream().mapToInt(String::length).sum();
IO.println(letters + " letras");
IO.println(IntStream.range(0, 5).boxed().toList());
IO.println(IntStream.rangeClosed(1, 10).filter(n -> n % 3 == 0).sum());
IntSummaryStatistics stats = IntStream.of(4, 8, 15, 16, 23, 42).summaryStatistics();
IO.println(stats.getMin() + " a " + stats.getMax() + ", média " + stats.getAverage());

🧪 **Experimente:** calcule 1 + 2 + … + 100 com `IntStream.rangeClosed`. Depois o produto de 1 a 5 com `.reduce(1, (a, b) -> a * b)`.

## 4. Operações terminais: obtendo uma resposta

A operação terminal decide o que sai:

| Operação | Devolve |
|---|---|
| `count()` | quantos itens (um `long`) |
| `sum()`, `average()`, `max()` | em streams de números (`mapToInt`, `mapToDouble`) |
| `anyMatch(teste)`, `allMatch(teste)` | `true` ou `false` |
| `toList()` | uma lista |
| `collect(Collectors.joining(", "))` | uma String só |

`average()` e `max()` devolvem um **Optional**, porque um stream vazio não tem média: `orElse(0)` dá um valor reserva.

In [ ]:
List<Double> totals = List.of(120.0, 35.5, 60.0, 15.0);
IO.println(totals.stream().filter(t -> t >= 50).count() + " pedidos grandes");
IO.println("Média: " + totals.stream().mapToDouble(t -> t).average().orElse(0));
IO.println("Algum acima de 100? " + totals.stream().anyMatch(t -> t > 100));
IO.println("Todos acima de 10? " + totals.stream().allMatch(t -> t > 10));
IO.println(totals.stream().map(t -> "R$" + t).collect(Collectors.joining(", ")));

🧪 **Experimente:** calcule a média de uma lista vazia (`List.<Double>of()`) com e sem o `orElse(0)`. Depois ache o maior total com `max()`.

## 5. Streams são preguiçosos e de uso único

Duas surpresas. As etapas intermediárias **não fazem nada** até uma operação terminal rodar: um pipeline sem ela nunca roda. E um stream só pode ser usado **uma vez**: para fazer uma segunda pergunta, comece de novo com um `.stream()` novo.

In [ ]:
List<Integer> values = List.of(1, 2, 3);
values.stream().map(v -> {
    IO.println("mapeando " + v);     // nunca impresso: não há operação terminal
    return v * 2;
});

Stream<Integer> once = values.stream().filter(v -> v > 1);
IO.println(once.count());
try {
    IO.println(once.count());        // o segundo uso
} catch (IllegalStateException e) {
    IO.println("Erro: " + e.getMessage());
}

🧪 **Experimente:** acrescente `.toList()` depois do primeiro `map` e rode: agora as linhas "mapeando" aparecem.

## 6. Laços ou streams?

Streams brilham quando a tarefa se lê como uma linha de montagem: fique com alguns, transforme, junte ou some. Um laço é mais claro quando os passos dependem uns dos outros, quando você precisa parar no meio de um jeito complicado, ou quando muda coisas de fora pelo caminho. Erros comuns com streams:

- **Mudar a lista de origem** dentro de um stream: monte uma lista nova com `toList()`.
- **`Optional.get()` num resultado vazio:** use `orElse` ou `ifPresent`.
- **Esquecer a operação terminal:** o pipeline nunca roda (passo 5).

In [ ]:
List<Integer> values = List.of(3, 9, 4);
Optional<Integer> big = values.stream().filter(v -> v > 100).findFirst();
IO.println(big.isPresent() + " " + big.orElse(-1));
big.ifPresent(v -> IO.println("achei " + v));     // não imprime nada

🧪 **Experimente:** chame `big.get()` e leia a exceção. Depois troque o filtro por `v > 5`.

## Em resumo

Um **stream** roda um pipeline sobre uma coleção: uma origem (`lista.stream()`), quantas etapas intermediárias quiser, como `filter`, `map`, `sorted` e `limit`, e depois **uma operação terminal** que produz o resultado.

```java
void main() {
    List<Double> orders = List.of(120.0, 35.5, 60.0, 15.0, 80.0);
    List<Double> top = orders.stream()
            .filter(t -> t >= 50)
            .sorted(Comparator.reverseOrder())
            .limit(2)
            .toList();
    IO.println(top);                                   // [120.0, 80.0]
    double total = orders.stream().mapToDouble(t -> t).sum();
    OptionalDouble avg = orders.stream().mapToDouble(t -> t).average();
    IO.println(total + " " + avg.orElse(0));
    IO.println(IntStream.rangeClosed(1, 5).map(n -> n * n).sum());   // 55
}
```

Streams de números (`mapToInt`, `mapToDouble`, `IntStream.range`) acrescentam `sum`, `average` e `max`; `average` e `max` devolvem um Optional, porque um stream vazio não tem nenhum.

**Cuidado:** nada roda até a operação terminal, e um stream só pode ser usado uma vez.

## Leituras

- [Processing Data in Memory Using the Stream API](https://dev.java/learn/api/collections-and-streams/streams/map-filter-reduce/) (dev.java (Oracle), 9 min): A ideia de map-filter-reduce, depois como um pipeline de stream a expressa: mapear, filtrar e reduzir a um resultado.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Quadrados dos pares** · Fácil
- **Estatísticas com streams** · Fácil
- **Arrumar uma lista de nomes** · Médio
- **Os 3 maiores pontuadores** · Difícil